# 베이스라인 — 수위로 유량을 예측한다 (rating curve)

입력은 [`02-water-task-candidates.ipynb`](02-water-task-candidates.ipynb)의 선정 결과다.

## 질문

**처음 보는 관측소**에서, 수위(gage height)와 관측소 지형 정보만으로 유량(discharge)을
얼마나 맞힐 수 있는가.

수문학에서 관측소마다 `Q = a(H − h₀)^b` 꼴의 **rating curve**를 따로 구한다.
여기서 묻는 것은 그 곡선을 **관측소별로 맞추는 것이 아니라**, 여러 관측소에서 배운 관계가
새 관측소로 **일반화되는가**이다.

## 설계

| 항목 | 값 |
|---|---|
| 타깃 | `flow_cfs`(00060). 로그정규 꼬리가 길어 **`log1p` 변환 후 학습** |
| 피처 | 수위 · 유역면적(log) · 고도 · 위경도 · 주 |
| 분할 | 🔴 **관측소(`site_no`) 단위** 그룹 분할 |
| 주지표 | 로그 스케일 **MAE**(+ 원 스케일 중앙값 오차). R²는 보조 |
| 시드 | `RANDOM_STATE = 42` 고정 |

## 🔴 이 데이터의 상한을 먼저 적는다

02의 실측: 관측 **시간 범위가 약 2시간**이고, 관측소 내 수위 변동폭 중앙값이 **0.01 ft**,
변동이 전혀 없는 관측소가 **54개**였다.

⇒ 페어 행은 1,600여 개지만 **독립 표본은 관측소 수(≈146)에 가깝다.** 같은 관측소의 행들은
사실상 중복이다. 그래서 이 노트북은 **행 수를 성능의 근거로 쓰지 않고**, §유효 표본 검증에서
관측소별 1행으로 줄여 같은 평가를 반복한다 — **행을 늘려도 성능이 오르지 않으면** 그 행들이
정보를 더하지 않았다는 뜻이다.

## 🔴 비-목적

- **실무 유량 추정에 쓰지 않는다.** USGS의 공식 rating curve는 관측소별 현장 측정으로 만든다.
- **이 수치를 리포트에 인용하지 않는다** — 인용하려면 gold 모델 승격이 선행한다
  ([`analysis.md`](../docs/conventions/analysis.md) §4). 마지막 셀에 승격 트리거를 적어 둔다.

## 데이터 출처 · 이용 조건

**출처: U.S. Geological Survey (USGS) National Water Information System.**

**미국 공공영역(U.S. Public Domain)** 이라 재배포 제한이 없고, 출처 표기는 USGS가
*"we **ask** that proper credit be given"* 이라 **요청**하는 것이다(`must`가 아니다).
근거와 `미확인` 잔여는 [`security.md`](../docs/security.md) **§0-1**.

재식별 축이 성립하지 않아 **소규모 셀 마스킹은 비적용**이다(02 첫 셀 참조).
단 **산출물 경로 규칙은 데이터셋과 무관하게 지킨다** — 규칙이 데이터셋마다 갈리면
다음 사람이 매번 판단해야 하고, 그 판단이 틀리는 날이 온다.

## 산출 엔진

**집계·피처 = Spark Connect / 학습 = 호스트 scikit-learn.**

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv

ENV_PATH = Path.cwd().parent / ".env"
loaded = load_dotenv(ENV_PATH)
SPARK_REMOTE = os.environ.get("SPARK_REMOTE", "sc://localhost:15002")
parsed = urlparse(SPARK_REMOTE.split(";")[0].rstrip("/"))
via_ingress = (parsed.hostname or "localhost") not in ("localhost", "127.0.0.1")

# 이 노트북 전체를 지배하는 상수 — 한곳에 모은다.
RANDOM_STATE = 42
N_SPLITS = 5
SLUG = "water-discharge"  # 산출물 디렉터리 이름

print(f".env      : {ENV_PATH} ({'읽음' if loaded else '없음'})")
print(f"접속 대상 : {SPARK_REMOTE}")
print(f"경로      : {'TLS Ingress' if via_ingress else 'port-forward(폴백)'}")
if via_ingress and not os.environ.get("GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"):
    print("🔴 GRPC_DEFAULT_SSL_ROOTS_FILE_PATH 미설정 — CA 검증에 실패한다")
print(f"seed {RANDOM_STATE} / {N_SPLITS}-fold")

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote(SPARK_REMOTE).getOrCreate()
print("Spark", spark.version)

# 🔴 관측 경로 생존 확인 — 아래 모든 수치의 유효 조건 (02와 같은 이유).
#    존재하지 않는 테이블은 "view support" 라는 엉뚱한 에러를 내므로 실재부터 본다.
tables = [r[1] for r in spark.sql("show tables in iceberg.usgs_water").collect()]
assert "water_iv_raw" in tables, f"water_iv_raw가 없다 (있는 것: {tables})"

n_probe = spark.sql(
    "select count(*) as n from iceberg.usgs_water.water_iv_raw"
).collect()[0]["n"]
assert n_probe > 0, "water_iv_raw가 0건 — 여기서 멈춘다"
print(f"water_iv_raw {n_probe:,} 행 — 관측 경로 생존 확인")

## 코호트 — attrition 표

제외 조건마다 몇 행이 빠지는지 센다(`analysis.md` §4 필수). **제외 사유가 없는 제외는 하지 않는다.**
검산으로 `제외합 + 최종 == 전체`를 확인한다 — 없으면 표의 숫자가 어긋나도 아무도 모른다.

| # | 제외 | 이유 |
|---|---|---|
| 1 | 수위 결측 | 피처가 없다 |
| 2 | 유량 결측 | 라벨이 없다 |
| 3 | 유량 ≤ 0 | 로그 변환이 정의되지 않고, rating curve는 양의 유량을 전제한다 |
| 4 | 관측소 메타 부재 | `water_sites`에 없는 관측소 — 지형 피처를 만들 수 없다 |

In [ ]:
import pandas as pd

# 같은 관측소·시각의 수위/유량을 한 행으로 편 뒤 관측소 메타를 붙인다.
PIVOT = """
select
    site_no,
    date_time,
    max(case when parameter_cd = '00065'
             then try_cast(value as double) end) as gage_ft,
    max(case when parameter_cd = '00060'
             then try_cast(value as double) end) as flow_cfs
from iceberg.usgs_water.water_iv_raw
group by site_no, date_time
"""

FLAGGED = f"""
select
    p.*,
    try_cast(s.drain_area_va as double) as drain_area,
    try_cast(s.alt_va as double) as altitude,
    try_cast(s.dec_lat_va as double) as latitude,
    try_cast(s.dec_long_va as double) as longitude,
    s.state_cd,
    (p.gage_ft is null) as ex1_no_gage,
    (p.flow_cfs is null) as ex2_no_flow,
    (p.flow_cfs is not null and p.flow_cfs <= 0) as ex3_nonpositive,
    (s.site_no is null) as ex4_no_meta
from ({PIVOT}) as p
left join iceberg.usgs_water.water_sites as s on p.site_no = s.site_no
"""

_P1 = "not ex1_no_gage"
_P2 = f"{_P1} and not ex2_no_flow"
_P3 = f"{_P2} and not ex3_nonpositive"

ATTRITION_SQL = f"""
select
    count(*) as n0,
    sum(case when ex1_no_gage then 1 else 0 end) as d1,
    sum(case when {_P1} and ex2_no_flow then 1 else 0 end) as d2,
    sum(case when {_P2} and ex3_nonpositive then 1 else 0 end) as d3,
    sum(case when {_P3} and ex4_no_meta then 1 else 0 end) as d4,
    sum(case when {_P3} and not ex4_no_meta then 1 else 0 end) as n_final
from ({FLAGGED})
"""
a = spark.sql(ATTRITION_SQL).toPandas().iloc[0]

steps = [
    ("전체 관측소-시각", "-", int(a["n0"])),
    ("- 수위 결측", "피처 없음", int(a["d1"])),
    ("- 유량 결측", "라벨 없음", int(a["d2"])),
    ("- 유량 <= 0", "로그 변환 불가", int(a["d3"])),
    ("- 관측소 메타 부재", "지형 피처 없음", int(a["d4"])),
]
remaining = int(a["n0"])
rows = []
for i, (label, reason, drop) in enumerate(steps):
    if i:
        remaining -= drop
    rows.append(
        {
            "단계": label,
            "이유": reason,
            "제외": f"{drop:,}" if i else "-",
            "남은 N": f"{remaining:,}",
        }
    )
display(pd.DataFrame(rows))

N_FINAL = int(a["n_final"])
total_dropped = sum(int(a[k]) for k in ("d1", "d2", "d3", "d4"))
# 🔴 검산 — 표의 숫자들이 서로 맞는가.
assert total_dropped + N_FINAL == int(a["n0"]), (
    f"attrition 불일치: {total_dropped:,} + {N_FINAL:,} != {int(a['n0']):,}"
)
assert remaining == N_FINAL, "순차 계산과 최종 카운트가 어긋난다"
print(f"✅ 검산 통과 — 제외합 {total_dropped:,} + 최종 {N_FINAL:,} = {int(a['n0']):,}")
print(f"최종 코호트 N = {N_FINAL:,} (산출 엔진: Spark Connect)")

In [ ]:
# 🔴 유일한 반출 지점. 관측소-시각 단위 표이고 개인 데이터가 아니다.
FEATURE_SQL = f"""
select
    site_no,
    date_time,
    gage_ft,
    flow_cfs,
    drain_area,
    altitude,
    latitude,
    longitude,
    state_cd
from ({FLAGGED})
where {_P3} and not ex4_no_meta
"""
df = spark.sql(FEATURE_SQL).toPandas()

assert len(df) == N_FINAL, f"피처 행 {len(df):,} != 코호트 {N_FINAL:,}"
assert df["flow_cfs"].min() > 0, "유량에 0 이하가 남았다 — attrition이 안 걸렸다"

print(f"피처표: {df.shape[0]:,} 행, {df.shape[1]} 열")
print(f"관측소 : {df['site_no'].nunique()}개")
print(f"메모리 : {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print()
print("유량(cfs) 분포")
display(df["flow_cfs"].describe(percentiles=[0.1, 0.5, 0.9, 0.99]).to_frame())
ratio = df["flow_cfs"].max() / df["flow_cfs"].median()
print(f"최대/중앙값 = {ratio:,.0f}배 — 꼬리가 길다. 그대로 회귀하면 큰 값이")
print("손실을 지배하므로 **log1p 변환 후 학습**하고, 평가도 로그 스케일에서 한다.")

In [ ]:
import numpy as np

# 피처 엔지니어링은 최소로 — 베이스라인의 목적은 성능이 아니라 경로 검증이다.
# 유역면적도 꼬리가 길어 로그를 취한다(면적은 유량과 대략 멱함수 관계다).
df["log_drain"] = np.log1p(df["drain_area"])
df["is_state_a"] = (df["state_cd"] == df["state_cd"].mode().iloc[0]).astype(int)
df["y_log"] = np.log1p(df["flow_cfs"])

FEATURES = ["gage_ft", "log_drain", "altitude", "latitude", "longitude", "is_state_a"]

y = df["y_log"].to_numpy()
groups = df["site_no"].to_numpy()  # 🔴 분할 단위는 관측소다

print(f"피처 {len(FEATURES)}개: {FEATURES}")
print(f"타깃: log1p(flow_cfs)  범위 {y.min():.2f} ~ {y.max():.2f}")
print()
miss = df[FEATURES].isna().mean() * 100
display(miss[miss > 0].round(2).to_frame("결측률 %"))
if miss.sum() == 0:
    print("결측 없음 — 그래도 파이프라인에는 대체기를 둔다")
    print("(다음 수집에서 결측이 생겨도 조용히 깨지지 않게).")

In [ ]:
from sklearn.model_selection import GroupKFold

# 🔴 관측소 단위 분할. 행 단위로 나누면 같은 관측소의 거의 동일한 행이
#    train과 test에 갈라져 들어가고, 모델은 **그 관측소의 곡선을 외운다**.
#    02에서 관측소 내 수위 변동폭 중앙값이 0.01 ft였다 — 사실상 같은 행이다.
cv = GroupKFold(n_splits=N_SPLITS)
folds = list(cv.split(df, y, groups))

for k, (tr, te) in enumerate(folds):
    overlap = set(groups[tr]) & set(groups[te])
    assert not overlap, f"fold {k}: 관측소 {len(overlap)}개가 양쪽에 걸쳤다 — 분할 누수"

sizes = [(len(tr), len(te), len(set(groups[te]))) for tr, te in folds]
print(f"✅ {N_SPLITS}-fold 관측소 단위 분할 — 겹치는 관측소 0개")
for k, (n_tr, n_te, n_sites_te) in enumerate(sizes):
    print(f"  fold {k}: train {n_tr:>5} / test {n_te:>5}  (test 관측소 {n_sites_te}개)")
print()
print("🔴 test 관측소 수가 곧 **유효 표본 수**다. 행 수가 아니다.")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def make_prep(features, scale=True):
    """결측 중앙값 대체(+표시자) → (선택) 표준화. 모델은 포함하지 않는다."""
    steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        steps.append(("scale", StandardScaler()))
    return ColumnTransformer([("num", Pipeline(steps), features)], remainder="drop")


def evaluate(model_factory, frame, y_vec, fold_list, scale=True, label=""):
    """그룹 분할 교차검증. 한 번의 성공은 결론이 아니므로 **분산까지** 본다.

    그룹 정보는 이미 `fold_list`에 반영돼 있다(분할을 만든 쪽이 책임진다).
    반환 지표는 전부 **로그 스케일**이다(타깃이 log1p).
    `중앙 배수오차`만 원 스케일로 되돌려 "몇 배 틀렸나"로 읽는다.
    """
    mae, r2, ratios = [], [], []
    for tr, te in fold_list:
        pipe = Pipeline(
            [
                ("prep", make_prep(FEATURES, scale)),
                ("model", model_factory()),
            ]
        )
        pipe.fit(frame.iloc[tr], y_vec[tr])
        pred = pipe.predict(frame.iloc[te])
        mae.append(mean_absolute_error(y_vec[te], pred))
        r2.append(r2_score(y_vec[te], pred))
        ratios.append(float(np.median(np.exp(np.abs(y_vec[te] - pred)))))
    return {
        "모델": label,
        "MAE(log)": f"{np.mean(mae):.3f} ± {np.std(mae):.3f}",
        "R2(log)": f"{np.mean(r2):.3f} ± {np.std(r2):.3f}",
        "중앙 배수오차": f"{np.mean(ratios):.2f}x",
        "_mae": float(np.mean(mae)),
        "_r2": float(np.mean(r2)),
    }


MODELS = {
    "dummy(중앙값)": (lambda: DummyRegressor(strategy="median"), False),
    "Ridge": (lambda: Ridge(alpha=1.0), True),
    "HistGB": (lambda: HistGradientBoostingRegressor(random_state=RANDOM_STATE), False),
}

results = [
    evaluate(factory, df, y, folds, scale=scale, label=name)
    for name, (factory, scale) in MODELS.items()
]
for r in results:
    print(
        f"{r['모델']:<14} MAE(log) {r['MAE(log)']}   R2 {r['R2(log)']}"
        f"   {r['중앙 배수오차']}"
    )

display(pd.DataFrame(results).drop(columns=["_mae", "_r2"]))
print("\n🔴 **dummy가 기준선이다.** dummy보다 나아지지 않으면 피처가 아무것도")
print("   말하지 않은 것이고, R2가 음수면 평균보다 못한 것이다.")

In [ ]:
# 🔴 음성 대조 — 이 셀이 이 노트북에서 가장 중요한 검증이다.
#    타깃을 섞으면 성능이 **dummy 수준으로 떨어져야** 한다. 안 떨어지면
#    누수이거나 평가 코드가 틀린 것이다. 좋은 지표라는 '성공 신호'를
#    의심할 유일한 수단이다.
rng = np.random.default_rng(0)
y_shuffled = rng.permutation(y)

shuf = evaluate(
    lambda: Ridge(alpha=1.0),
    df,
    y_shuffled,
    folds,
    scale=True,
    label="타깃 셔플",
)
dummy_mae = next(r for r in results if r["모델"].startswith("dummy"))["_mae"]
best_mae = min(r["_mae"] for r in results)

print(f"dummy MAE(log)     : {dummy_mae:.3f}")
print(f"최고 모델 MAE(log) : {best_mae:.3f}")
print(f"타깃 셔플 MAE(log) : {shuf['MAE(log)']}   R2 {shuf['R2(log)']}")
print()
if shuf["_mae"] < dummy_mae * 0.9:
    print("🔴 셔플했는데도 dummy보다 낫다 — 누수 또는 평가 버그를 의심하라.")
    print("   여기서 멈추고 원인을 찾는다. 아래 수치는 신뢰할 수 없다.")
else:
    print("→ 셔플 시 신호 없음. 파이프라인이 타깃을 통해서만 배우고 있다.")
if shuf["_r2"] > 0.05:
    print(f"⚠️ 셔플 R2가 {shuf['_r2']:.3f}로 0보다 뚜렷이 크다 — 확인이 필요하다.")

## 🔴 유효 표본 검증 — 행을 늘리면 정말 좋아지는가

02에서 "행 1,600여 개지만 유효 표본은 관측소 수(≈146)에 가깝다"고 주장했다.
**주장은 검증해야 주장이 아니게 된다.**

관측소별로 **1행**(중앙값)으로 줄여 같은 평가를 반복한다.

- 성능이 **거의 같으면** → 늘어난 행들이 정보를 더하지 않았다. 유효 표본 주장이 맞다.
- 성능이 **뚜렷이 나빠지면** → 행들이 실제로 정보를 담고 있었다. 주장을 철회해야 한다.

어느 쪽이든 결과를 그대로 적는다.

In [ ]:
# 관측소별 1행으로 축약 — 수치는 중앙값, 메타는 첫 값(관측소 내 불변).
agg = df.groupby("site_no", as_index=False).agg(
    {
        "gage_ft": "median",
        "log_drain": "first",
        "altitude": "first",
        "latitude": "first",
        "longitude": "first",
        "is_state_a": "first",
        "y_log": "median",
    }
)
y_agg = agg["y_log"].to_numpy()
groups_agg = agg["site_no"].to_numpy()
folds_agg = list(GroupKFold(n_splits=N_SPLITS).split(agg, y_agg, groups_agg))

print(f"축약: {len(df):,}행 → {len(agg):,}행 (관측소 1행씩)")
print()

agg_results = [
    evaluate(factory, agg, y_agg, folds_agg, scale=scale, label=name)
    for name, (factory, scale) in MODELS.items()
]

compare = pd.DataFrame(
    [
        {
            "모델": full["모델"],
            f"MAE 전체({len(df):,}행)": full["MAE(log)"],
            f"MAE 축약({len(agg):,}행)": red["MAE(log)"],
            "차이": f"{red['_mae'] - full['_mae']:+.3f}",
        }
        for full, red in zip(results, agg_results, strict=True)
    ]
)
display(compare)

best_full = min(r["_mae"] for r in results)
best_agg = min(r["_mae"] for r in agg_results)
delta = (best_agg - best_full) / best_full * 100
print(f"최고 모델 기준 MAE 변화: {delta:+.1f}%")
if abs(delta) < 15:
    print("→ 행을 10배 줄여도 성능이 거의 같다.")
    print("  **유효 표본 주장이 실측으로 지지된다** — 늘어난 행은 정보를")
    print("  더하지 않았다. 행 수를 성능의 근거로 쓰면 안 된다.")
else:
    print("→ 성능이 뚜렷이 달라졌다. 02의 '유효 표본 ≈ 관측소 수' 주장을")
    print("  철회하고, 관측소 내 변동이 실제로 정보를 담았는지 다시 본다.")

In [ ]:
import json
import shutil
import subprocess
from datetime import datetime, timezone

import joblib
import sklearn

# 🔴 산출물은 **저장소 밖**에 둔다.
#    이 데이터셋은 공개 관측치라 DUA 대상이 아니지만, **경로 규칙은 데이터셋과
#    무관하게 지킨다** — 규칙이 데이터셋마다 갈리면 다음 사람이 매번 판단해야 하고
#    그 판단이 틀리는 날이 온다. 같은 코드로 MIMIC-IV에 돌아갈 수도 있다.
#
#    ⚠️ scripts/worker_path_guard.py는 **워커의 쓰기만** 본다.
#       Jupyter 커널은 그 가드 밖이라, 아래 검사가 커널 축의 **유일한 방어**다.
git_bin = shutil.which("git")


def _git(*args, default=""):
    """git을 읽기 전용으로 부른다. 실패하면 default."""
    if not git_bin:
        return default
    out = subprocess.run(  # noqa: S603
        [git_bin, *args], cwd=Path.cwd(), capture_output=True, text=True, check=False
    )
    return out.stdout.strip() or default


# 🔴 저장소 루트는 cwd에서 유추하지 않고 **git에게 묻는다** — nbconvert를 repo
#    루트에서 돌리면 `Path.cwd().parent`가 $HOME이 되어 경계 검사의 기준이 어긋난다.
_toplevel = _git("rev-parse", "--show-toplevel")
REPO_ROOT = Path(_toplevel).resolve() if _toplevel else Path.cwd().parent.resolve()

ARTIFACT_DIR = (
    Path(os.environ.get("DATA_EXTRACT_DIR", Path.home() / "extracts")).expanduser()
    / "ml"
    / SLUG
)
resolved = ARTIFACT_DIR.resolve()
if resolved == REPO_ROOT or REPO_ROOT in resolved.parents:
    msg = (
        f"산출물 경로가 저장소 안이다: {resolved} (저장소 루트 {REPO_ROOT})\n"
        "저장소 밖($DATA_EXTRACT_DIR)에만 쓴다."
    )
    raise RuntimeError(msg)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 최고 모델을 전체 데이터로 다시 적합해 저장한다.
best_name = min(results, key=lambda r: r["_mae"])["모델"]
best_factory, best_scale = MODELS[best_name]
best = Pipeline(
    [
        ("prep", make_prep(FEATURES, best_scale)),
        ("model", best_factory()),
    ]
)
best.fit(df, y)

# 실험 추적의 실체는 이 JSON 하나다(MLflow 없음 — 상주 서비스 0개).
# **다시 만들 수 있을 만큼** 적는다: 시드·분할·버전·커밋·엔진.
metrics = {
    "task": SLUG,
    "question": "predict discharge (00060) from gage height (00065) + site terrain",
    "target": "log1p(flow_cfs)",
    "features": FEATURES,
    "n_rows": len(df),
    "n_sites": int(df["site_no"].nunique()),
    "effective_sample_note": (
        "관측 시간 범위가 약 2시간이라 관측소 내 행은 사실상 중복이다. "
        "유효 표본은 행 수가 아니라 관측소 수에 가깝다(§유효 표본 검증에서 실측)."
    ),
    "random_state": RANDOM_STATE,
    "cv": f"GroupKFold(n_splits={N_SPLITS}) on site_no",
    "results_full": [
        {k: v for k, v in r.items() if not k.startswith("_")} for r in results
    ],
    "results_site_aggregated": [
        {k: v for k, v in r.items() if not k.startswith("_")} for r in agg_results
    ],
    "shuffle_control_mae_log": shuf["_mae"],
    "dummy_mae_log": dummy_mae,
    "saved_model": best_name,
    "engines": {
        "aggregation": f"Spark Connect {spark.version}",
        "training": "host scikit-learn",
    },
    "versions": {
        "scikit-learn": sklearn.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
    "repo_commit": _git("rev-parse", "HEAD", default="unknown"),
    "generated_at": datetime.now(timezone.utc).isoformat(),
}
(ARTIFACT_DIR / "metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False)
)
joblib.dump(best, ARTIFACT_DIR / "model.joblib")

print(f"저장소 루트: {REPO_ROOT}")
print(f"저장 위치  : {ARTIFACT_DIR}")
for f in sorted(ARTIFACT_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size:,} B)")
print(f"\n저장된 모델: {best_name}")

## 한계 — 비우지 않는다

1. **표본이 작다.** 유효 표본은 관측소 수(≈146)에 가깝고, 그중 fold 하나의 test는 30개 남짓이다.
   지표의 표준편차를 반드시 함께 읽어야 한다.
2. 🔴 **관측 시간 범위가 약 2시간**이다. 계절·유량 조건의 다양성이 전혀 없어,
   갈수기·홍수기에 이 관계가 유지되는지는 **이 데이터로는 알 수 없다**.
   rating curve는 원래 고유량에서 크게 휘는데 그 구간이 표본에 없다.
3. **관측소별 곡선을 배우지 않았다** — 이 설계는 의도적으로 새 관측소 일반화를 물었다.
   실무에서 특정 관측소의 유량을 추정할 때는 그 관측소의 rating curve가 훨씬 정확하다.
4. **지형 피처가 얕다.** 유역면적·고도·위경도뿐이고, 하도 경사·조도·단면 형상 같은
   실제 결정 변수는 데이터에 없다.
5. **`value` 컬럼이 string으로 적재**돼 있어 `try_cast`로 변환했다. 변환 실패는 0건이었으나,
   USGS가 품질 코드(Ice·Bkw 등)를 값에 넣는 경우 조용히 NULL이 되므로 다음 수집에서는
   다시 확인해야 한다.
6. **하이퍼파라미터 탐색이 없다.** 베이스라인이므로 기본값만 썼다.

## 🔴 이 수치를 리포트에 쓰려면 — 승격이 선행 조건

`analysis.md` §1·§4에 따라 **노트북 수치는 결론의 근거가 되지 못한다.**
`docs/analyses/`에 인용하려면 먼저:

1. 위 피벗·코호트 SQL을 `usgs_water__rating_pairs`(`tags=['gold']`, `materialized='table'`)로 승격
2. grain을 모델 설명 첫 줄에 명시(`1행 = site_no × date_time`) + `schema.yml`에 유니크 테스트
3. 코호트·피처 정의를 `schema.yml` `description`에 기록
4. **그 다음에** 리포트를 쓴다

**승격 트리거(조건, 시점 아님)**: 같은 피벗 SQL을 3회째 조회하는 순간, 또는
이 수치를 인용하는 리포트를 쓰기로 결정하는 순간 — 둘 중 먼저 오는 쪽.

In [ ]:
spark.stop()
print("세션 종료")
print()
print("🔴 컴퓨트 회수 — 터미널에서:")
print("   kubectl scale deploy/spark-connect --replicas=0")
print("   kubectl get pods -l spark-role=executor   # 비어야 한다")
print("   (기동 시 '떠 있었다'를 기록했을 때만 이 '비었다'가 회수를 뜻한다)")